# GRACE H3 — Colab CPU preparation

Bắt đầu từ notebooks/full-pipeline.ipynb trên local. Chọn Runtime → Change runtime type → accelerator **None**. Upload hai zip đã đóng gói vào MyDrive/GRACE-H3-transfer:

1. grace-h3-cpu-inputs.zip — code/config/dataset/sáu embedding sets/graph và candidate cache.
2. grace-h3-joern-runtime.zip — Joern 4.0.623 C/C++, notebook cài Java21.

Không cần GPU/Kaggle, không gọi LLM, không dùng Vertex/ADC. Nếu chọn H1+H3, H1 heads phải được import ở local trước khi tạo input bundle; chúng đã nằm trong input.

Fit gate → calibration riêng → threshold selection riêng → route → prepare GRACE cho inspect. full_reference=true prepare thêm full cohort để so benchmark. Hoàn tất tải cpu-output.zip về local để gọi LLM.

In [ ]:
from pathlib import Path,PurePosixPath
import os,sys,json,zipfile,hashlib,shutil,subprocess
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[name]='4'
USE_GOOGLE_DRIVE=True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRANSFER=Path('/content/drive/MyDrive/GRACE-H3-transfer')
else:
    TRANSFER=Path('/content/transfer') # Upload the two files using Colab Files pane.
TRANSFER.mkdir(parents=True,exist_ok=True)
ROOT=Path('/content/GRACE-H3-CalibratedCascade')
INPUTS=TRANSFER/'grace-h3-cpu-inputs.zip'
JOERN=TRANSFER/'grace-h3-joern-runtime.zip'
CHECKPOINT=TRANSFER/'grace-h3-cpu-checkpoint.zip'
OUTPUT=TRANSFER/'grace-h3-cpu-output.zip'
for p in [INPUTS,JOERN]:
    if not p.is_file(): raise FileNotFoundError(p)
print('RAM GiB:',round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3,2))

Mounted at /content/drive
RAM GiB: 12.67


## 1. Nhập snapshot, runtime và checkpoint

Zip được copy lên /content rồi giải nén với checksum. Các file trùng được reuse; nội dung khác bị chặn để tránh trộn experiment. Dùng runtime và thư mục transfer mới nếu đổi code/config.

In [ ]:
ROOT.mkdir(parents=True,exist_ok=True)
# Copy a single archive to native disk before extracting thousands of members.
local_inputs=Path('/content/grace-h3-cpu-inputs.zip')
shutil.copyfile(INPUTS,local_inputs)
with zipfile.ZipFile(local_inputs) as z:
    manifest=json.loads(z.read('bundle_manifest.json'))
    if manifest['schema']!='grace_h3_bundle_v1' or manifest['kind']!='cpu_inputs': raise ValueError('Wrong CPU inputs')
    names=[e['path'] for e in manifest['entries']]
    if len({n.casefold() for n in names})!=len(names): raise ValueError('Duplicate paths')
    if sorted(z.namelist())!=sorted(names+['bundle_manifest.json']): raise ValueError('Unexpected members')
    for e in manifest['entries']:
        name=e['path'];rel=PurePosixPath(name)
        if rel.is_absolute() or '..' in rel.parts or ':' in name or chr(92) in name: raise ValueError('Unsafe path')
        target=ROOT/name;target.resolve().relative_to(ROOT)
        if target.exists():
            h=hashlib.sha256()
            with target.open('rb') as src:
                for chunk in iter(lambda:src.read(1024*1024),b''): h.update(chunk)
            if target.stat().st_size==e['bytes'] and h.hexdigest()==e['sha256']: continue
            raise ValueError(f'Different existing input: {name}; use a new runtime')
        target.parent.mkdir(parents=True,exist_ok=True)
        temp=target.with_name(target.name+'.bundle.tmp');h=hashlib.sha256();size=0
        with z.open(name) as src,temp.open('wb') as dst:
            for chunk in iter(lambda:src.read(1024*1024),b''):
                h.update(chunk);size+=len(chunk);dst.write(chunk)
        if size!=e['bytes'] or h.hexdigest()!=e['sha256']: raise ValueError('Input checksum mismatch')
        temp.chmod(e.get('mode',0o644) & 0o777);os.replace(temp,target)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from grace_h3.portable import import_bundle,create_bundle
from grace_h1.io import read_json
print(import_bundle(ROOT,JOERN,'joern_runtime'))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'cpu_checkpoint'))
print('CPU inputs, Joern and optional checkpoint imported')

{'kind': 'joern_runtime', 'copied': 219, 'reused_identical': 0}
CPU inputs, Joern and optional checkpoint imported


## 2. Dependency CPU và Java21

Không cài Torch/CUDA/CodeT5/SDK LLM. Joern chỉ chạy khi thiếu graph; mặc định hai workers, heap2GB mỗi JVM. Cấu hình source/data/thresholds trong input đã khóa; muốn đổi thì sửa ở local và tạo lại bundle.

In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt')])
subprocess.check_call(['apt-get','update','-qq'])
subprocess.check_call(['apt-get','install','-y','-qq','openjdk-21-jre-headless'])
java=sorted(Path('/usr/lib/jvm').glob('java-21-openjdk-*'))
if not java: raise RuntimeError('Java21 not found')
os.environ['JAVA_HOME']=str(java[0]);os.environ['PATH']=str(java[0]/'bin')+os.pathsep+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools'/'joern'/'joern-cli')
subprocess.check_call([str(java[0]/'bin'/'java'),'-version'])
config=read_json(ROOT/'configs'/'experiment.json')
print(json.dumps(config,indent=2))

{
  "datasets": [
    "devign"
  ],
  "split": "test",
  "full_reference": true,
  "retrieval_arm": "baseline_whitening",
  "graph_workers": 2,
  "inference_workers": 4,
  "bootstrap": 1000,
  "gate": {
    "C": 1,
    "class_weight": null,
    "max_iter": 1000,
    "seed": 42,
    "calibration_fraction": 0.5,
    "low_grid": [
      0.01,
      0.025,
      0.05,
      0.075,
      0.1,
      0.15,
      0.2,
      0.25,
      0.3
    ],
    "high_grid": [
      0.7,
      0.8,
      0.9,
      0.95,
      0.975,
      0.99
    ],
    "enable_direct_high": false,
    "max_lost_positive_fraction": 0.01,
    "min_high_precision": 0.98,
    "max_high_false_positive_fraction": 0.01,
    "family_alpha": 0.05
  }
}


## 3. Fit gate và đọc policy trước phần graph

Train duy nhất cho logistic/normalization. Validation bỏ code trùng train và nhóm mâu thuẫn, chia calibration/selection riêng; không đọc test labels để chọn thresholds. Mỗi code hash chỉ giữ một bản trong phần fit/calibration/selection. Test/routing cohort vẫn full.

low=0 nghĩa là không có threshold đạt constraint, toàn bộ inspect. Đây là kết quả hợp lệ; không nới constraint sau khi nhìn test để tạo call saving. Các bound là nominal, không bảo đảm ngoài mẫu hoặc qua domain shift. Checkpoint gate hoàn tất được lưu ngay.

In [ ]:
from grace_h3.gate import fit_gate
from grace_h3.workflow import prepare_all,selected_jobs
from threadpoolctl import threadpool_limits
import pandas as pd

def save_checkpoint():
    info=create_bundle(ROOT,ROOT/'bundles'/'grace-h3-cpu-checkpoint.zip','cpu_checkpoint')
    temp=CHECKPOINT.with_name(CHECKPOINT.name+'.tmp')
    shutil.copyfile(info['path'],temp);os.replace(temp,CHECKPOINT)
    print('Checkpoint saved:',CHECKPOINT,'bytes:',info['bytes'],flush=True)

with threadpool_limits(limits=4):
    for dataset in config['datasets']:
        gate=fit_gate(ROOT,dataset,config['gate'])
        print(dataset,'counts:',gate['counts'],'low:',gate['policy']['low'],'high:',gate['policy']['high'])
        display(pd.DataFrame(gate['policy']['selection_low']))
        if gate['policy']['selection_high']: display(pd.DataFrame(gate['policy']['selection_high']))
        print('Held-out selection calibration:',gate['selection_calibrated'])
        save_checkpoint()

devign counts: {'train_total': 21854, 'fit_unique_unambiguous': 21762, 'validation_total': 2732, 'validation_eligible_unique': 2724, 'calibration': 1362, 'selection': 1362} low: 0.075 high: None


,threshold,count,fn,positive_denominator,lost_positive_fraction_upper,eligible
0,0.010,0,0,592,0.008734,False
1,0.025,0,0,592,0.008734,False
2,0.050,3,0,592,0.008734,True
3,0.075,5,0,592,0.008734,True
4,0.100,11,1,592,0.012283,False
5,0.150,38,4,592,0.020874,False
6,0.200,85,12,592,0.040044,False
7,0.250,156,34,592,0.086221,False
8,0.300,261,67,592,0.150168,False


Held-out selection calibration: {'brier': 0.22390390605447444, 'nll': 0.6371405199989496, 'ece_10_equal_width': 0.02520240067659107, 'n': 1362, 'prevalence': 0.434654919236417}
Checkpoint saved: /content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-checkpoint.zip bytes: 84403


## 4. Route và chuẩn bị frozen prompts

Chỉ query inspect cần retrieval/Joern cho cascade. full_reference=true tạo thêm các query còn lại để có baseline paired; đó là chi phí nghiên cứu, không tính thành deployed cascade. Hai nhánh inspect/reference phải có exact prompts giống nhau.

Checkpoint được lưu ở mốc 500 graph khi có cache mới và cuối mỗi run/job. Interrupt cố lưu trong finally; pool có thể phải chờ các task Joern đang chạy/đã submit. Runtime bị xóa đột ngột chỉ giữ checkpoint Drive cuối cùng. Chạy một coordinator trên mỗi thư mục transfer.

In [ ]:
try:
    with threadpool_limits(limits=4):
        prepared=prepare_all(ROOT,config,checkpoint_callback=save_checkpoint)
finally:
    save_checkpoint()
display(pd.DataFrame(prepared))
for folder,job in selected_jobs(ROOT,config):
    print(job['dataset'],'branches:',job['branch_counts'],'LLM query ratio:',job['branch_counts'].get('inspect',0)/job['n'])

devign/test common candidates 250/2726 (0s)
devign/test common candidates 500/2726 (0s)
devign/test common candidates 750/2726 (0s)
devign/test common candidates 1000/2726 (0s)
devign/test common candidates 1250/2726 (0s)
devign/test common candidates 1500/2726 (0s)
devign/test common candidates 1750/2726 (0s)
devign/test common candidates 2000/2726 (0s)
devign/test common candidates 2250/2726 (0s)
devign/test common candidates 2500/2726 (0s)
devign/baseline_whitening: 12179 unique graph features required
graphs 500/12179
Checkpoint saved: /content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-checkpoint.zip bytes: 29559237
graphs 1000/12179
graphs 1500/12179
graphs 2000/12179
graphs 2500/12179
graphs 3000/12179
graphs 3500/12179
Checkpoint saved: /content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-checkpoint.zip bytes: 29597945
graphs 4000/12179
Checkpoint saved: /content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-checkpoint.zip bytes: 29926182
graphs 4500/12179
Checkpoint saved: /

,dataset,path
0,devign,runs/h3/devign/3c90db3984588497


devign branches: {'inspect': 2726, 'skip_low': 6} LLM query ratio: 0.9978038067349927


## 5. Xuất kết quả → quay local

cpu-output.zip chứa gate NPZ/manifest, policy/routing, frozen GRACE runs, selection và cache mới. Không đóng gói lại graph có sẵn trong input. Không có credential hay LLM responses.

Tải grace-h3-cpu-output.zip về bundles/ local rồi Run All full-pipeline.ipynb lại. Local import/verify trước preflight → inference → evaluation toàn cohort. cpu-checkpoint.zip dùng resume Colab, không thay cho output hoàn tất.

In [ ]:
print('Ready:',len(selected_jobs(ROOT,config)),'dataset jobs')
info=create_bundle(ROOT,ROOT/'bundles'/'grace-h3-cpu-output.zip','cpu_output')
temp=OUTPUT.with_name(OUTPUT.name+'.tmp');shutil.copyfile(info['path'],temp);os.replace(temp,OUTPUT)
print('DOWNLOAD:',OUTPUT,'bytes:',info['bytes'],'sha256:',info['sha256'])
from IPython.display import FileLink,display
display(FileLink(str(OUTPUT)))
# Download from Google Drive or Files pane before the runtime ends.

Ready: 1 dataset jobs
DOWNLOAD: /content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-output.zip bytes: 74775943 sha256: 801d4d29e7ca125dfd145b464f5e8de8003e1d4a440fe75057ab00ed46cd8df7


/content/drive/MyDrive/GRACE-H3-transfer/grace-h3-cpu-output.zip